# Week 2 — Prompt engineering

**Deliverable:** notebook with zero/few-shot patterns + **IT ticket classifier** (JSON output).

Uses **[Ollama](https://ollama.com)** locally (no API key). Pull a model: `ollama pull llama3.1:8b`

### Before you run — pick the right kernel in Cursor

Top-right **Select Kernel** → **`Week-02 .venv (3.13.9)`** (`week02-venv`).

Do **not** use `Python 3 (ipykernel)` or **Anaconda base** — those use the wrong Python and cells will fail or hang.

**First-time setup (terminal):** `uv pip install -r requirements.txt` (do not use `%pip` — uv venv has no pip)

**Run order after restart:** Dependencies → Connectivity → **Setup** → Zero-shot → Few-shot

Optional `.env`: `OLLAMA_MODEL=llama3.1:8b`, `OLLAMA_HOST=http://localhost:11434`

## 0) Dependencies

In [6]:
# Verify deps (install once in terminal: uv pip install -r requirements.txt)
import importlib
import sys

missing = []
for pkg, mod in [("httpx", "httpx"), ("python-dotenv", "dotenv"), ("ollama", "ollama")]:
    try:
        importlib.import_module(mod)
    except ModuleNotFoundError:
        missing.append(pkg)

if missing:
    raise ModuleNotFoundError(
        f"Missing: {', '.join(missing)}. In Week-02 folder run:\n"
        f"  uv pip install -r requirements.txt\n"
        f"Then restart kernel: {sys.executable}"
    )
print("Dependencies OK (httpx, python-dotenv, ollama)")

Dependencies OK (httpx, python-dotenv, ollama)


In [7]:
# Quick connectivity check — Ollama must be running locally
import os
import httpx
from dotenv import load_dotenv

load_dotenv()
host = os.getenv("OLLAMA_HOST", "http://localhost:11434").rstrip("/")
model = os.getenv("OLLAMA_MODEL", "llama3.1:8b")

r = httpx.get(f"{host}/api/tags", timeout=10)
print("Ollama host:", host)
print("HTTP status:", r.status_code)
if r.status_code != 200:
    raise RuntimeError("Start Ollama (app or `ollama serve`), then re-run this cell.")

names = [m["name"] for m in r.json().get("models", [])]
print("Installed models:", names or "(none — run: ollama pull llama3.1:8b)")
if model not in names and f"{model}:latest" not in names:
    print(f"Warning: {model!r} not pulled yet. Run: ollama pull {model}")
else:
    print(f"Ready — will use model: {model}")

Ollama host: http://localhost:11434
HTTP status: 200
Installed models: ['nomic-embed-text:latest', 'llama3.1:8b']
Ready — will use model: llama3.1:8b


## 1) Zero-shot vs few-shot

In [8]:
# Setup — run this once after restart (before zero-shot / few-shot)
import os, json, re
import ollama
from dotenv import load_dotenv

load_dotenv()
if os.getenv("OLLAMA_HOST"):
    os.environ["OLLAMA_HOST"] = os.getenv("OLLAMA_HOST")

MODEL = os.getenv("OLLAMA_MODEL", "llama3.1:8b")
print("Using model:", MODEL)


def ollama_chat(messages, max_tokens=200):
    return ollama.chat(
        model=MODEL,
        messages=messages,
        options={"num_predict": max_tokens},
    )


def parse_json_response(text: str) -> dict:
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[-1]
        if text.lstrip().startswith("json"):
            text = text.split("\n", 1)[-1]
        if text.rstrip().endswith("```"):
            text = text.rsplit("```", 1)[0]
        text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if not match:
            raise
        return json.loads(match.group(0))


print("Setup OK — run zero-shot cell next")


Using model: llama3.1:8b
Setup OK — run zero-shot cell next


In [9]:
# Zero-shot classifier
zero_shot = [
    {
        "role": "user",
        "content": (
            "Classify this ticket as access|network|hardware|application|other. "
            "Reply JSON only.\n\nTitle: VPN drops every 10 minutes\n"
            "Body: Wi-Fi fine, only VPN client disconnects."
        ),
    }
]

resp = ollama_chat(zero_shot, max_tokens=200)
print(resp["message"]["content"])

```json
{
    "classification": "network"
}
```


In [10]:
FEW_SHOT = """You are an IT triage assistant. Output JSON ONLY with keys:
category, confidence (low|med|high), reason (one sentence).

Categories: access, network, hardware, application, other.

Examples:
Ticket: Title: Need MFA reset | Body: locked out of email
{"category":"access","confidence":"high","reason":"Account lockout/MFA is an access control issue."}

Ticket: Title: Printer jam | Body: Tray 2 keeps jamming
{"category":"hardware","confidence":"high","reason":"Physical printer failure."}

Now classify:
"""

ticket = """Title: SQL timeout in reporting job
Body: SSRS report runs > 30s then fails with timeout; DBA says blocking spikes at 9am."""

msg = ollama_chat(
    [{"role": "user", "content": FEW_SHOT + ticket}],
    max_tokens=250,
)
raw = msg["message"]["content"]
print(raw)

try:
    data = parse_json_response(raw)
    print("Parsed:", data)
except json.JSONDecodeError:
    print("Model did not return strict JSON — tighten the prompt or add a repair step.")


```
{
  "category": "application",
  "confidence": "med",
  "reason": "Application error is likely due to an issue with the reporting job or its configuration."
}
```
Parsed: {'category': 'application', 'confidence': 'med', 'reason': 'Application error is likely due to an issue with the reporting job or its configuration.'}


## 2) Homework starter — three prompts for your role

Edit the next cell with **your** tasks.

In [11]:
MY_PROMPTS = [
    {"name": "task_1_summarise_meeting", "prompt": "TODO: paste a real prompt you would use at work."},
    {"name": "task_2_draft_email", "prompt": "TODO"},
    {"name": "task_3_checklist", "prompt": "TODO"},
]

for p in MY_PROMPTS:
    print("===", p["name"], "===")
    print(p["prompt"])


=== task_1_summarise_meeting ===
TODO: paste a real prompt you would use at work.
=== task_2_draft_email ===
TODO
=== task_3_checklist ===
TODO


## 3) Deliverable checklist

- [ ] Few-shot classifier returns JSON for at least 10 sample tickets
- [ ] Three real-world prompts documented above